## ETL Gold – Tablas para análisis (clima, astronomía y riesgos)

### Propósito
Construir tablas **Gold** en Delta (listas para consumo analítico) a partir de tablas **Silver**, aplicando:
- deduplicación por `ingestion_time` (último dato disponible)
- métricas derivadas (rango térmico, heladas, GDD, ventana de siembra)
- agregados diarios desde datos horarios (picos de lluvia/viento y horas fúngicas)
- tabla final de riesgos para dashboard

### Entradas (Silver)
- `weather_daily_silver`
- `weather_daily_silver_forecast`
- `weather_hourly_silver`
- `weather_astronomy_daily_silver`

### Salidas (Gold, Delta managed tables)
- `daily_actual_gold` (1 fila por `city + date`)
- `daily_forecast_gold` (1 fila por `city + date`)
- `astronomy_daily_gold` (1 fila por `city + date`)
- `agricultural_risk_metrics` (1 fila por `city + date`)

### Métricas principales generadas
- En `daily_actual_gold`:
  - `temp_range`, `frost_risk`, `dry_day`, `gdd_daily`
- En `daily_forecast_gold`:
  - `temp_range`, `frost_risk`, `sowing_window`
- En `astronomy_daily_gold`:
  - `day_length_hours` (derivado de `sunrise_ts` y `sunset_ts`)
- En `agricultural_risk_metrics` (a partir de agregados horarios):
  - `max_hourly_precip`, `max_hourly_wind`, `storm_risk`
  - (según agregado horario) `fungal_hours` como base para riesgo fúngico

### Notas
- La granularidad de todas las tablas Gold es diaria (`city + date`), orientada a consumo en dashboards.
- Para detalle de columnas y lógica de deduplicación, ver `documentacion.md`.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max as spark_max, sum as spark_sum, when, lit, greatest,to_date, to_timestamp,concat_ws
from pyspark.sql import functions as F

In [0]:
spark = SparkSession.builder \
    .appName("Weather Gold ETL") \
    .getOrCreate()

df_daily_actual = spark.read.format("delta").table("weather_daily_silver")

df_daily_forecast = spark.read.format("delta").table("weather_daily_silver_forecast")

df_hourly = spark.read.format("delta").table("weather_hourly_silver")

df_astronomy = spark.read.format("delta").table("weather_astronomy_daily_silver")

# Normalizar formato de timestamp (reemplazar guiones con dos puntos en la parte de hora)
df_daily_actual = df_daily_actual.withColumn(
    "ingestion_time", 
    to_timestamp(F.regexp_replace(col("ingestion_time"), "T(\\d{2})-(\\d{2})-(\\d{2})", "T$1:$2:$3"))
)

df_daily_forecast = df_daily_forecast.withColumn(
    "ingestion_time", 
    to_timestamp(F.regexp_replace(col("ingestion_time"), "T(\\d{2})-(\\d{2})-(\\d{2})", "T$1:$2:$3"))
).withColumn(
    "date", to_date(col("date"))
)

df_hourly = df_hourly.withColumn(
    "ingestion_time", 
    to_timestamp(F.regexp_replace(col("ingestion_time"), "T(\\d{2})-(\\d{2})-(\\d{2})", "T$1:$2:$3"))
)

df_astronomy = df_astronomy.withColumn(
    "ingestion_time", 
    to_timestamp(F.regexp_replace(col("ingestion_time"), "T(\\d{2})-(\\d{2})-(\\d{2})", "T$1:$2:$3"))
).withColumn(
    "sunrise_ts", to_timestamp(col("sunrise_ts"))
).withColumn(
    "sunset_ts", to_timestamp(col("sunset_ts"))
)

df_astronomy = df_astronomy.withColumn(
    "moonrise_ts", to_timestamp(concat_ws(" ", col("date"), col("moonrise")), "yyyy-MM-dd hh:mm a")
).withColumn(
    "moonset_ts", to_timestamp(concat_ws(" ", col("date"), col("moonset")), "yyyy-MM-dd hh:mm a")
)

df_daily_actual.printSchema()
df_daily_forecast.printSchema()
df_hourly.printSchema()
df_astronomy.printSchema()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
# Deduplicar daily_actual
window_daily = Window.partitionBy("city", "date").orderBy(F.col("ingestion_time").desc())
df_daily_actual_clean = df_daily_actual.withColumn("rank", F.row_number().over(window_daily)) \
                                       .filter(F.col("rank") == 1).drop("rank")

# Deduplicar daily_forecast
window_forecast = Window.partitionBy("city", "date").orderBy(F.col("ingestion_time").desc())
df_daily_forecast_clean = df_daily_forecast.withColumn("rank", F.row_number().over(window_forecast)) \
                                           .filter(F.col("rank") == 1).drop("rank")

# Deduplicar astronomy
window_astronomy = Window.partitionBy("city", "date").orderBy(F.col("ingestion_time").desc())
df_astronomy_clean = df_astronomy.withColumn("rank", F.row_number().over(window_astronomy)) \
                                 .filter(F.col("rank") == 1).drop("rank")


In [0]:
df_hourly_agg = df_hourly.groupBy("city", "date").agg(
    F.max("precip_mm").alias("max_hourly_precip"),
    F.max("wind_kph").alias("max_hourly_wind"),
    F.sum(F.when((F.col("humidity") > 80) & (F.col("temp_c").between(15,25)), 1).otherwise(0)).alias("fungal_hours")
)


In [0]:
df_daily_actual_gold = df_daily_actual_clean.withColumn(
    "temp_range", F.col("maxtemp_c") - F.col("mintemp_c")
).withColumn(
    "frost_risk",
    F.when((F.col("mintemp_c") < 2) & (F.col("maxwind_kph") < 15), "Alto")
     .when(F.col("mintemp_c") < 2, "Moderado")
     .otherwise("Bajo")
).withColumn(
    "dry_day", F.col("totalprecip_mm") == 0
).withColumn(
    "gdd_daily", F.greatest(F.lit(0), (F.col("maxtemp_c")+F.col("mintemp_c"))/2 - F.lit(10)) # Base para maíz
)


In [0]:
df_daily_forecast_gold = df_daily_forecast_clean.withColumn(
    "temp_range", F.col("maxtemp_c") - F.col("mintemp_c")
).withColumn(
    "frost_risk",
    F.when((F.col("mintemp_c") < 2) & (F.col("maxwind_kph") < 15), "Alto")
     .when(F.col("mintemp_c") < 2, "Moderado")
     .otherwise("Bajo")
).withColumn(
    "sowing_window",
    F.when((F.col("avghumidity") > 60) & (F.col("daily_chance_of_rain") < 30) & (F.col("totalprecip_mm") < 10), "Óptimo")
     .when((F.col("avghumidity") > 50) & (F.col("daily_chance_of_rain") < 50), "Marginal")
     .otherwise("No apto")
)


In [0]:
df_astronomy_gold = df_astronomy_clean.withColumn(
    "day_length_hours", 
    (F.col("sunset_ts").cast("long") - F.col("sunrise_ts").cast("long")) / 3600
)


In [0]:
df_agri_risk = df_daily_actual_gold.alias("d") \
    .join(df_hourly_agg.alias("h"), ["city", "date"], "left") \
    .select(
        F.col("d.city"),
        F.col("d.date"),
        F.col("temp_range"),
        F.col("frost_risk"),
        F.col("dry_day"),
        F.col("gdd_daily"),
        F.col("h.max_hourly_precip"),
        F.col("h.max_hourly_wind"),
        # Riesgo de tormenta basado en hourly
        F.when((F.col("h.max_hourly_precip") > 10) & (F.col("h.max_hourly_wind") > 40), "Alto")
         .when((F.col("h.max_hourly_precip") > 5) & (F.col("h.max_hourly_wind") > 30), "Moderado")
         .otherwise("Bajo").alias("storm_risk")
    )


In [0]:
print("===== daily_actual_gold =====")
display(df_daily_actual_gold.limit(5))

print("\n===== daily_forecast_gold =====")
display(df_daily_forecast_gold.limit(5))

print("\n===== astronomy_daily_gold =====")
display(df_astronomy_gold.limit(5))

print("\n===== agricultural_risk_metrics =====")
display(df_agri_risk.limit(5))

In [0]:
print("Escribiendo daily_actual_gold...")
df_daily_actual_gold.write.format("delta").mode("overwrite").saveAsTable("daily_actual_gold")

print("Escribiendo daily_forecast_gold...")
df_daily_forecast_gold.write.format("delta").mode("overwrite").saveAsTable("daily_forecast_gold")

print("Escribiendo astronomy_daily_gold...")
df_astronomy_gold.write.format("delta").mode("overwrite").saveAsTable("astronomy_daily_gold")

print("Escribiendo agricultural_risk_metrics...")
df_agri_risk.write.format("delta").mode("overwrite").saveAsTable("agricultural_risk_metrics")

print("\n✅ Todas las tablas Gold han sido escritas exitosamente")